# ⚡ Household Electricity Consumption Analysis

**Data Analysis and Visualization (DAV) Lab Project**

This notebook demonstrates the complete DAV pipeline on a simulated household electricity dataset:

1. Import libraries
2. Load dataset
3. Understand the dataset
4. Data cleaning
5. Descriptive statistics
6. Exploratory data analysis
7. Visualization
8. Correlation analysis
9. High-consumption analysis
10. Findings and conclusion

> ⚠️ The dataset is simulated. Room-wise values are **estimated usage categories**. High-consumption flags identify *unusual patterns worth investigating* — not confirmed wastage.

## 1. Import Libraries

In [ ]:
import sys, os
# Allow the notebook to import analysis.py from the project root
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import analysis

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)

print('Libraries imported successfully.')

## 2. Load Dataset

If the CSV does not exist yet, run `python generate_dataset.py` first (from the project root).

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'electricity_consumption.csv')

df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

## 3. Understand the Dataset

Check the shape, column types, and a sample of records before any cleaning.

In [ ]:
print('Shape:', df_raw.shape)
df_raw.info()

In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())
print()
print('Duplicate rows:', df_raw.duplicated().sum())
print('Duplicate Record_IDs:', df_raw['Record_ID'].duplicated().sum())

## 4. Data Cleaning

Cleaning steps performed by `analysis.clean_data()`:

1. Parse `Date` as datetime.
2. Fill missing numeric values with the **column median** (robust to outliers) instead of deleting rows.
3. Drop exact duplicate rows and duplicated `Record_ID`s.
4. Validate ranges: non-negative power/energy, hours 0–23, plausible voltage (200–250 V).

In [ ]:
df = analysis.clean_data(df_raw)
print('Rows before cleaning:', len(df_raw))
print('Rows after cleaning :', len(df))
df.head()

## 5. Descriptive Statistics

Headline numbers for the whole dataset — total, average, median, spread, peak hour, and the weekday/weekend split.

In [ ]:
stats = analysis.calculate_statistics(df)
for key, value in stats.items():
    if isinstance(value, float):
        print(f'{key:18s}: {value:,.3f}')
    else:
        print(f'{key:18s}: {value}')

In [ ]:
df.describe().round(3)

## 6 & 7. Exploratory Data Analysis and Visualization

### 6.1 Hourly consumption — when does the household use electricity?

In [ ]:
hourly = analysis.calculate_hourly_consumption(df)

plt.figure(figsize=(10, 4))
plt.plot(hourly['Hour'], hourly['Avg_Power_kW'], marker='o', color='#3498db')
plt.title('Average Power Consumption by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Avg Power (kW)')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

peak_hours_ranked = analysis.calculate_peak_hours(df)
print('Top 5 highest-consumption hours:')
print(peak_hours_ranked[['Hour', 'Avg_Power_kW', 'Classification']].head(5).to_string(index=False))

### 6.2 Peak-hour classification

Hours are classified with data-driven thresholds (not manual judgement):
- **Peak** — avg consumption ≥ 75th percentile of hourly averages
- **Low** — avg consumption ≤ 25th percentile
- **Normal** — in between

In [ ]:
colors = peak_hours_ranked['Classification'].map({'Peak': '#e74c3c', 'Normal': '#3498db', 'Low': '#2ecc71'})

plt.figure(figsize=(10, 4))
plt.bar(peak_hours_ranked['Hour'], peak_hours_ranked['Avg_Power_kW'], color=colors)
plt.title('Hour Classification: Peak / Normal / Low')
plt.xlabel('Hour')
plt.ylabel('Avg Power (kW)')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### 6.3 Daily consumption trend

In [ ]:
daily = analysis.calculate_daily_consumption(df)

plt.figure(figsize=(12, 4))
plt.plot(daily['Date'], daily['Total_kWh'], color='#3498db', linewidth=1)
plt.title('Daily Total Consumption')
plt.xlabel('Date')
plt.ylabel('Total Energy (kWh)')
plt.tight_layout()
plt.show()

### 6.4 Weekday vs weekend

In [ ]:
comparison = analysis.calculate_weekday_weekend(df)
display(comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(comparison['Day_Type'], comparison['Avg_Energy_kWh'], color=['#3498db', '#e67e22'])
axes[0].set_title('Average Consumption: Weekday vs Weekend')
axes[0].set_ylabel('Avg Energy (kWh)')
axes[1].bar(comparison['Day_Type'], comparison['Total_Energy_kWh'], color=['#3498db', '#e67e22'])
axes[1].set_title('Total Consumption: Weekday vs Weekend')
axes[1].set_ylabel('Total Energy (kWh)')
plt.tight_layout()
plt.show()

hourly_split = analysis.calculate_hourly_by_day_type(df)
plt.figure(figsize=(10, 4))
for col, color in [('Weekday', '#3498db'), ('Weekend', '#e67e22')]:
    if col in hourly_split.columns:
        plt.plot(hourly_split['Hour'], hourly_split[col], marker='o', label=col, color=color)
plt.title('Average Consumption by Hour: Weekday vs Weekend')
plt.xlabel('Hour')
plt.ylabel('Avg Power (kW)')
plt.xticks(range(0, 24))
plt.legend()
plt.tight_layout()
plt.show()

### 6.5 Monthly consumption

In [ ]:
monthly = analysis.calculate_monthly_consumption(df)
display(monthly.round(3))

plt.figure(figsize=(11, 4))
plt.plot(monthly['Month'], monthly['Total_kWh'], marker='o', color='#9b59b6')
plt.title('Monthly Total Consumption')
plt.xlabel('Month')
plt.ylabel('Total Energy (kWh)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

top = monthly.loc[monthly['Total_kWh'].idxmax()]
low = monthly.loc[monthly['Total_kWh'].idxmin()]
print(f"Highest-consumption month : {top['Month']} ({top['Total_kWh']:.1f} kWh)")
print(f"Lowest-consumption month  : {low['Month']} ({low['Total_kWh']:.1f} kWh)")

### 6.6 Day × Hour heatmap

The main visualization of the project — average consumption for every combination of weekday and hour.

In [ ]:
matrix = analysis.calculate_day_hour_matrix(df)

plt.figure(figsize=(14, 5))
sns.heatmap(matrix, cmap='YlOrRd', annot=True, fmt='.2f', linewidths=0.5)
plt.title('Average Consumption: Day of Week × Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Day')
plt.tight_layout()
plt.show()

### 6.7 Room-wise estimated consumption

> These are **estimated usage categories** from the simulated dataset, not metered sub-circuits.

In [ ]:
rooms = analysis.calculate_room_consumption(df)
display(rooms.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(rooms['Room'], rooms['Total_kWh'], color=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'])
axes[0].set_title('Total Estimated Consumption by Category')
axes[0].set_ylabel('Total Energy (kWh)')

axes[1].pie(rooms['Total_kWh'], labels=rooms['Room'], autopct='%1.1f%%',
            colors=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'], startangle=90,
            wedgeprops=dict(width=0.45))
axes[1].set_title('Share of Estimated Consumption')
plt.tight_layout()
plt.show()

### 6.8 Consumption distribution

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df['Power_Consumption_kW'], bins=50, color='#3498db', edgecolor='white')
plt.title('Distribution of Power Consumption')
plt.xlabel('Power (kW)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

### 8.1 Voltage vs Power

> ⚠️ **Correlation does not imply causation.** The correlation coefficient only indicates the strength and direction of the association *in this dataset*.

In [ ]:
corr = analysis.voltage_power_correlation(df)
print(f'Pearson correlation (Voltage vs Power): {corr:.3f}')

plt.figure(figsize=(8, 5))
plt.scatter(df['Voltage_V'], df['Power_Consumption_kW'], alpha=0.3, s=10, color='#3498db')
plt.title('Voltage vs Power Consumption')
plt.xlabel('Voltage (V)')
plt.ylabel('Power (kW)')
plt.tight_layout()
plt.show()

### 8.2 Full correlation heatmap

In [ ]:
corr_matrix = analysis.calculate_correlations(df)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Heatmap: Electrical Quantities and Usage Categories')
plt.tight_layout()
plt.show()

## 9. High-Consumption Analysis

### 9.1 Unusually high-consumption records

Records above the statistical threshold are **Potential High-Consumption Periods** — unusual patterns that may be worth investigating. They are **not confirmed electricity wastage**: high consumption can be legitimate (guests, seasonal appliance use, etc.).

Two methods are available:
- **IQR**: upper bound = Q3 + 1.5 × IQR (robust to outliers)
- **Std**: threshold = Mean + 2 × Standard Deviation

In [ ]:
df_flagged, threshold, method_desc = analysis.detect_high_consumption(df, method='iqr')
n_high = int(df_flagged['High_Consumption'].sum())
print(f'Method: {method_desc}')
print(f'Threshold: {threshold:.3f} kWh')
print(f'Flagged records: {n_high} ({n_high / len(df_flagged) * 100:.2f}% of all records)')

plt.figure(figsize=(12, 4))
colors = df_flagged['High_Consumption'].map({True: '#e74c3c', False: '#3498db'})
plt.scatter(df_flagged['Date'], df_flagged['Energy_Consumption_kWh'], c=colors, s=10, alpha=0.5)
plt.axhline(threshold, color='black', linestyle='--', label=f'Threshold {threshold:.2f} kWh')
plt.title('Normal vs Potential High-Consumption Records')
plt.xlabel('Date')
plt.ylabel('Energy (kWh)')
plt.legend()
plt.tight_layout()
plt.show()

### 9.2 Unusually high-consumption days

Daily totals above **Mean + 2 × Standard Deviation** are flagged as unusual days.

In [ ]:
daily_flagged, day_threshold, day_method = analysis.detect_high_consumption_days(df, method='std')
n_days = int(daily_flagged['High_Consumption'].sum())
print(f'Method: {day_method} | Threshold: {day_threshold:.2f} kWh | Unusual days: {n_days}')

plt.figure(figsize=(12, 4))
colors = daily_flagged['High_Consumption'].map({True: '#e74c3c', False: '#3498db'})
plt.scatter(daily_flagged['Date'], daily_flagged['Total_kWh'], c=colors, s=15)
plt.axhline(day_threshold, color='black', linestyle='--', label=f'Threshold {day_threshold:.2f} kWh')
plt.title('Daily Consumption with Unusual Days Highlighted')
plt.xlabel('Date')
plt.ylabel('Daily Total (kWh)')
plt.legend()
plt.tight_layout()
plt.show()

if n_days > 0:
    print('Unusual days:')
    print(daily_flagged.loc[daily_flagged['High_Consumption'], ['Date', 'Total_kWh']].to_string(index=False))

## 10. Findings and Conclusion

All insights below are computed dynamically from the dataset.

In [ ]:
for i, insight in enumerate(analysis.generate_insights(df), start=1):
    # strip markdown bold markers for plain console output
    print(f'{i}. {insight.replace(chr(42), "")}')

### Conclusion

This notebook walked through the complete DAV pipeline:

```text
DATA → DATA CLEANING → EXPLORATORY ANALYSIS → STATISTICAL ANALYSIS
     → VISUALIZATION → INTERACTIVE DASHBOARD → INSIGHTS
```

- The household shows a clear **evening peak** and **night trough**, with different rhythms on weekends.
- Seasonal variation identifies the highest- and lowest-consumption months.
- Voltage and power show only a weak association, and **correlation does not prove causation**.
- A small fraction of records/days are statistically unusual — **potential high-consumption periods** worth investigating, not confirmed wastage.

The same analysis functions (`analysis.py`) power the interactive Streamlit dashboard (`app.py`), where all of these results can be explored with live filters.